# Qwen3-VL Invoice Extraction v5 — Document-Aware VLM Pipeline

VLM-only invoice extraction using Qwen3-VL-4B / Qwen3-VL-8B.

**Core flow:**

`PDF/image → page analysis → duplicate-copy removal → page-aware VLM extraction → table-aware line items → financial-summary mapping → canonical JSON`

Duplicate `raw_fields` are intentionally retained for audit. Important values must also populate canonical JSON fields.

No OCR, FastAPI, Uvicorn, ngrok, or backend mathematical/business validation is included.

## 1. Install dependencies

In [ ]:
!pip install -q -U "transformers>=4.57.0" accelerate bitsandbytes pydantic python-dateutil "qwen-vl-utils[decord]>=0.0.14" "pillow>=10.1" pymupdf


## 2. Imports

In [ ]:
import base64
import copy
import io
import json
import os
import re
import tempfile
import time
import datetime
import asyncio
import threading
import traceback
from datetime import timedelta
from typing import Optional, List, Dict, Any, Union

import torch
from PIL import Image
from pydantic import BaseModel, Field, ValidationError, create_model
from dateutil import parser as dateparser
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

torch.manual_seed(0)

# Concurrency lock for safe GPU inference
inference_lock = asyncio.Lock()

# Unique Request ID generator (VLM-YYYYMMDD-HHMMSS-NNNN)
_vlm_request_counter = 0
_vlm_counter_lock = threading.Lock()

def generate_vlm_request_id() -> str:
    global _vlm_request_counter
    with _vlm_counter_lock:
        _vlm_request_counter += 1
        now_str = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        return f"VLM-{now_str}-{_vlm_request_counter:04d}"


## 3. Load Qwen3-VL — 4B / 8B switch

This notebook is designed to test the extraction pipeline, not the final deployment quantization choice.

- `MODEL_SIZE = "4B"` is the default.
- Set `MODEL_SIZE = "8B"` to benchmark Qwen3-VL-8B on the same invoice.
- `USE_4BIT = True` is only for local/Colab testing.
- For the production/deployment configuration, set `USE_4BIT = False`.
- Qwen3-VL uses a 16×16 vision patch with a 2×2 spatial merge, giving an effective 32×32 spatial unit. This notebook therefore uses 32×32 when converting a visual-token budget into a pixel budget.

In [ ]:
MODEL_SIZE = "4B"          # "4B" or "8B"
USE_4BIT = True            # TESTING ONLY. Set False for production/deployment validation.
MODEL_NAME_VL = {
    "4B": "Qwen/Qwen3-VL-4B-Instruct",
    "8B": "Qwen/Qwen3-VL-8B-Instruct",
}[MODEL_SIZE]

# Qwen3-VL visual budget.
# The model's vision config uses patch_size=16 and spatial_merge_size=2,
# which corresponds to a 32x32 effective spatial unit.
VISION_UNIT = 32
MIN_VISUAL_TOKENS_FULL = 256
MAX_VISUAL_TOKENS_FULL = 1536
MAX_VISUAL_TOKENS_CROP = 2048

MIN_PIXELS_FULL = MIN_VISUAL_TOKENS_FULL * VISION_UNIT * VISION_UNIT
MAX_PIXELS_FULL = MAX_VISUAL_TOKENS_FULL * VISION_UNIT * VISION_UNIT
MIN_PIXELS_CROP = MIN_VISUAL_TOKENS_FULL * VISION_UNIT * VISION_UNIT
MAX_PIXELS_CROP = MAX_VISUAL_TOKENS_CROP * VISION_UNIT * VISION_UNIT

# Qwen's base model is BF16. Use BF16 when the GPU supports it; otherwise use FP16.
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
        bnb_4bit_use_double_quant=True,
    )
    hf_model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_NAME_VL,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=COMPUTE_DTYPE,
    )
else:
    hf_model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_NAME_VL,
        device_map="auto",
        torch_dtype=COMPUTE_DTYPE,
    )

processor = AutoProcessor.from_pretrained(
    MODEL_NAME_VL,
    min_pixels=MIN_PIXELS_FULL,
    max_pixels=MAX_PIXELS_FULL,
)

hf_model.eval()

print("=" * 70)
print("MODEL:", MODEL_NAME_VL)
print("MODEL SIZE:", MODEL_SIZE)
print("4-bit TEST MODE:", USE_4BIT)
print("COMPUTE DTYPE:", COMPUTE_DTYPE)
print("FULL IMAGE MAX PIXELS:", MAX_PIXELS_FULL)
print("CROP MAX PIXELS:", MAX_PIXELS_CROP)
print("=" * 70)

## 4. Invoice schema — extraction-first

The schema is designed so that visible invoice concepts have a real structured destination.
`raw_fields` is still retained as a duplicate/audit representation.

Important additions for this test version:
- `document_type` distinguishes invoice / purchase order / other when visible.
- `unit` is kept for line items such as Month, Lot, Man-days, etc.
- `hsn_code` remains for compatibility, while `hsn_sac_code` and `code_type` preserve the source meaning.
- `line_amount` represents a generic row amount when the table has an `Amount` column; it is not silently treated as taxable amount.
- `taxable_amount` is populated only when the source/table explicitly identifies a taxable value.

In [ ]:
from typing import Optional, List, Dict, Any
from pydantic import BaseModel, Field, ConfigDict

class BankDetails(BaseModel):
    account_holder_name: Optional[str] = None
    account_number: Optional[str] = None
    ifsc_code: Optional[str] = None
    bank_name: Optional[str] = None
    branch: Optional[str] = None
    upi_id: Optional[str] = None

class LineItem(BaseModel):
    # What is visibly billed in the actual item/service row.
    description: Optional[str] = None

    # Code fields: keep hsn_code for backward compatibility.
    hsn_code: Optional[str] = None
    hsn_sac_code: Optional[str] = None
    code_type: Optional[str] = None  # HSN / SAC

    quantity: Optional[float] = None
    unit: Optional[str] = None
    unit_price: Optional[float] = None
    discount: Optional[float] = None

    # Only use taxable_amount when the source column/label explicitly means taxable/assessable value.
    taxable_amount: Optional[float] = None

    # Generic amount/value printed for the billed row.
    line_amount: Optional[float] = None

    gst_rate: Optional[float] = None
    cgst_rate: Optional[float] = None
    cgst_amount: Optional[float] = None
    sgst_rate: Optional[float] = None
    sgst_amount: Optional[float] = None
    igst_rate: Optional[float] = None
    igst_amount: Optional[float] = None

    # Explicit total for the row when the invoice prints one.
    total: Optional[float] = None

    # Duplicate/source evidence is allowed and intentionally retained.
    raw_fields: Dict[str, Any] = Field(default_factory=dict)
    additional_fields: Dict[str, Any] = Field(default_factory=dict)


class InvoiceSchema(BaseModel):
    document_type: Optional[str] = None

    invoice_number: Optional[str] = None
    invoice_date: Optional[str] = None
    due_date: Optional[str] = None
    po_number: Optional[str] = None

    vendor_name: Optional[str] = None
    vendor_address: Optional[str] = None
    vendor_gstin: Optional[str] = None
    vendor_pan: Optional[str] = None
    vendor_cin: Optional[str] = None
    vendor_phone: Optional[str] = None
    vendor_email: Optional[str] = None

    customer_name: Optional[str] = None
    customer_address: Optional[str] = None
    customer_gstin: Optional[str] = None
    customer_pan: Optional[str] = None

    payment_terms: Optional[str] = None
    bank_details: Optional[BankDetails] = None

    line_items: List[LineItem] = Field(default_factory=list)

    # Financial concepts remain separate. Do not force Taxable Value into subtotal.
    subtotal: Optional[float] = None
    taxable_amount: Optional[float] = None
    discount_total: Optional[float] = None
    tax_total: Optional[float] = None
    cgst_total: Optional[float] = None
    sgst_total: Optional[float] = None
    igst_rate: Optional[float] = None
    igst_total: Optional[float] = None
    adjustment: Optional[float] = None
    shipping_charges: Optional[float] = None
    other_charges: Optional[float] = None
    round_off: Optional[float] = None
    total_amount: Optional[float] = None
    currency: Optional[str] = None

    additional_fields: Dict[str, Any] = Field(default_factory=dict)
    raw_fields: Dict[str, Any] = Field(default_factory=dict)

_header_field_defs = {
    name: (field.annotation, field)
    for name, field in InvoiceSchema.model_fields.items()
    if name != "line_items"
}
InvoiceHeaderSchema = create_model("InvoiceHeaderSchema", **_header_field_defs)

CORE_FIELDS = [
    f for f in InvoiceSchema.model_fields.keys()
    if f not in ("line_items", "additional_fields", "raw_fields")
]

print("Schema loaded.")
print("Invoice fields:", len(InvoiceSchema.model_fields))
print("Line-item fields:", len(LineItem.model_fields))

## 5. Extraction prompts

In [ ]:
# ---------------------------------------------------------------------------
# V5 extraction contract
# ---------------------------------------------------------------------------

HEADER_SCHEMA_COMPACT = {
    "document_type": None,
    "invoice_number": None,
    "invoice_date": None,
    "due_date": None,
    "po_number": None,
    "vendor_name": None,
    "vendor_address": None,
    "vendor_gstin": None,
    "vendor_pan": None,
    "vendor_cin": None,
    "vendor_phone": None,
    "vendor_email": None,
    "customer_name": None,
    "customer_address": None,
    "customer_gstin": None,
    "customer_pan": None,
    "payment_terms": None,
    "bank_details": {
        "account_holder_name": None,
        "account_number": None,
        "ifsc_code": None,
        "bank_name": None,
        "branch": None,
        "upi_id": None,
    },
    "subtotal": None,
    "taxable_amount": None,
    "discount_total": None,
    "tax_total": None,
    "cgst_total": None,
    "sgst_total": None,
    "igst_rate": None,
    "igst_total": None,
    "adjustment": None,
    "shipping_charges": None,
    "other_charges": None,
    "round_off": None,
    "total_amount": None,
    "currency": None,
    "additional_fields": {},
    "raw_fields": {},
}

LINE_ITEM_SCHEMA_COMPACT = {
    "description": None,
    "hsn_code": None,
    "hsn_sac_code": None,
    "code_type": None,
    "quantity": None,
    "unit": None,
    "unit_price": None,
    "discount": None,
    "taxable_amount": None,
    "line_amount": None,
    "gst_rate": None,
    "cgst_rate": None,
    "cgst_amount": None,
    "sgst_rate": None,
    "sgst_amount": None,
    "igst_rate": None,
    "igst_amount": None,
    "total": None,
    "raw_fields": {},
    "additional_fields": {},
}

FOOTER_SCHEMA_COMPACT = {
    "subtotal": None,
    "taxable_amount": None,
    "discount_total": None,
    "tax_total": None,
    "cgst_total": None,
    "sgst_total": None,
    "igst_rate": None,
    "igst_total": None,
    "adjustment": None,
    "shipping_charges": None,
    "other_charges": None,
    "round_off": None,
    "total_amount": None,
    "currency": None,
    "bank_details": {
        "account_holder_name": None,
        "account_number": None,
        "ifsc_code": None,
        "bank_name": None,
        "branch": None,
        "upi_id": None,
    },
    "additional_fields": {},
    "raw_fields": {},
}

HEADER_SCHEMA_JSON = json.dumps(HEADER_SCHEMA_COMPACT, indent=2)
LINE_ITEM_SCHEMA_JSON = json.dumps(LINE_ITEM_SCHEMA_COMPACT, indent=2)
FOOTER_SCHEMA_JSON = json.dumps(FOOTER_SCHEMA_COMPACT, indent=2)

SYSTEM_PROMPT_HEADER = """You are an exact visual invoice extraction engine.

Read the supplied page directly. Extract only information that is visibly present.

Rules:
- Never guess or invent missing values.
- Copy identifiers exactly: invoice number, GSTIN, PAN, PO number, bank identifiers.
- Preserve printed dates and numbers.
- Seller/vendor is the issuing party; buyer/customer is the billed-to/party receiving the invoice.
- A field must be populated only when the page provides evidence for that field.
- DO NOT treat a Purchase Order document number as an invoice number.
- DO NOT populate subtotal unless the source explicitly labels a value as subtotal.
- DO NOT populate taxable_amount unless the source explicitly labels a value as taxable/assessable/taxable value, or an explicit taxable-value column exists.
- DO NOT populate tax_total/CGST/SGST/IGST totals unless an explicit tax amount is visible.
        - If the source explicitly states IGST with a rate, populate igst_rate and igst_total.
        - If the source explicitly labels a monetary value as Adjustment, populate adjustment; do not relabel it as other_charges.
- DO NOT calculate totals.
- If information is not visible, return null.
- Keep useful duplicate/source label-value evidence in raw_fields.
- Do not extract line-item rows in this pass.
- Return exactly one JSON object and nothing else.

Schema:
""" + HEADER_SCHEMA_JSON

SYSTEM_PROMPT_LINE_ITEMS = """You are an exact visual invoice table-extraction engine.

Read the supplied invoice page/crop directly.

A LINE ITEM is a real billed goods/service/usage row belonging to the item's table.
It is NOT simply any row inside a table.

Include only rows that represent an actual billed product/service/usage item.

NEVER treat these as line items:
- subtotal
- taxable value summary
- CGST/SGST/IGST rows
- GST/tax summary rows
- total / grand total
- amount payable
- invoice summary rows
- notes, declarations, payment instructions, bank rows

Other rules:
- Preserve row order.
- Do not invent quantity, rate, code, tax or total.
- Use the visible column headers to map values.
- An Amount column maps to line_amount, not automatically taxable_amount.
- A Taxable Value column maps to taxable_amount.
- Preserve unit separately from quantity.
- HSN and SAC must be distinguished using the visible label.
- If a tax label such as IGST @ 18% appears inside the same item row, keep the product/service description as the description and put 18 in igst_rate.
- Do not populate generic gst_rate when the visible tax is explicitly IGST/CGST/SGST.
- Return [] when no genuine line-item row is present.
- Return exactly one JSON array and nothing else.

Schema for each row:
""" + LINE_ITEM_SCHEMA_JSON

SYSTEM_PROMPT_FOOTER = """You are an exact visual invoice-summary extraction engine.

Read the supplied lower-page crop directly.

Extract only explicit invoice-level financial summary, tax summary, bank/payment and related document metadata.

Rules:
- Never infer or calculate.
- Taxable Value is not automatically Subtotal.
- Total Invoice Value / Total / Grand Total is not the same as Taxable Value.
- Preserve explicit IGST/CGST/SGST amounts and rates.
        - An explicit Adjustment line maps to adjustment, not other_charges.
- Do not output line items.
- Do not move account numbers into IFSC or IFSC into account numbers.
- If something is not visible, return null.
- Keep raw source labels in raw_fields when present.
- Return exactly one JSON object and nothing else.

Schema:
""" + FOOTER_SCHEMA_JSON

print("V5 extraction contract loaded.")

## 6. VLM generation and JSON parsing

In [ ]:
# ---------------------------------------------------------------------------
# V5 page-aware extraction, duplicate-copy detection and safe normalization
# ---------------------------------------------------------------------------

TABLE_TOP_RATIO = 0.08
TABLE_BOTTOM_RATIO = 0.90
FOOTER_TOP_RATIO = 0.25
FOOTER_BOTTOM_RATIO = 1.00

DUPLICATE_PAGE_SIMILARITY_THRESHOLD = 0.995
ENABLE_DUPLICATE_PAGE_DETECTION = True


def save_vertical_crop(image_path: str, top_ratio: float, bottom_ratio: float, tag: str) -> str:
    img = Image.open(image_path).convert("RGB")
    w, h = img.size
    top = max(0, min(h - 1, int(h * top_ratio)))
    bottom = max(top + 1, min(h, int(h * bottom_ratio)))
    crop = img.crop((0, top, w, bottom))
    crop_path = f"/content/qwen_invoice_crop_{tag}_{os.getpid()}_{int(time.time()*1000)}.png"
    crop.save(crop_path, format="PNG")
    return crop_path


def _prepare_inputs(messages):
    text_prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(
        messages,
        image_patch_size=16
    )
    return processor(
        text=[text_prompt],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
        do_resize=False,
    ).to(hf_model.device)


def _generate_raw(messages, max_new_tokens: int):
    inputs = _prepare_inputs(messages)
    pad_id = processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id

    with torch.no_grad():
        out = hf_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=pad_id,
        )

    input_len = inputs["input_ids"].shape[1]
    return processor.batch_decode(
        out[:, input_len:],
        skip_special_tokens=True
    )[0]


def _balanced_json_block(text: str, opener: str, closer: str):
    text = strip_code_fences(text or "")
    start = text.find(opener)
    if start < 0:
        return None

    depth = 0
    in_string = False
    escape = False

    for i in range(start, len(text)):
        ch = text[i]

        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue

        if ch == '"':
            in_string = True
        elif ch == opener:
            depth += 1
        elif ch == closer:
            depth -= 1
            if depth == 0:
                return text[start:i + 1]

    return None


def _parse_json_object(raw_text):
    parsed = safe_json_parse(raw_text or "")
    if isinstance(parsed, dict):
        return parsed

    candidate = _balanced_json_block(raw_text, "{", "}")
    if candidate:
        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, dict):
                return parsed
        except Exception:
            pass

    return None


def _parse_json_array(raw_text):
    try:
        parsed = json.loads(strip_code_fences(raw_text or ""))
        if isinstance(parsed, list):
            return parsed
    except Exception:
        pass

    candidate = _balanced_json_block(raw_text, "[", "]")
    if candidate:
        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass

    return None


def _repair_unparseable_json_with_model(raw_text: str, expected: str):
    if not raw_text or expected not in {"object", "array"}:
        return None

    schema_hint = "{}" if expected == "object" else "[]"

    messages = [{
        "role": "user",
        "content": [{
            "type": "text",
            "text": (
                "Repair ONLY the JSON syntax of the following malformed output. "
                "Do not add, remove, infer, calculate, normalize, or change any value. "
                f"Return ONLY valid JSON of type {expected}. Example: {schema_hint}.\n\n"
                "RAW OUTPUT:\n" + raw_text
            )
        }]
    }]

    try:
        text_prompt = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        inputs = processor(
            text=[text_prompt],
            padding=True,
            return_tensors="pt",
        ).to(hf_model.device)

        with torch.no_grad():
            out = hf_model.generate(
                **inputs,
                max_new_tokens=min(4096, max(512, len(raw_text) // 2)),
                do_sample=False,
                pad_token_id=processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id,
            )

        input_len = inputs["input_ids"].shape[1]
        repaired = processor.batch_decode(
            out[:, input_len:],
            skip_special_tokens=True
        )[0]

        return _parse_json_object(repaired) if expected == "object" else _parse_json_array(repaired)
    except Exception:
        return None


def _normalize_line_item_v5(item):
    if not isinstance(item, dict):
        return {}

    out = {}
    raw = dict(item.get("raw_fields")) if isinstance(item.get("raw_fields"), dict) else {}
    extra = dict(item.get("additional_fields")) if isinstance(item.get("additional_fields"), dict) else {}

    aliases = {
        "description": ["description", "item", "item description", "product", "service", "particulars", "details", "goods"],
        "hsn_code": ["hsn_code", "hsn code", "hsn"],
        "hsn_sac_code": ["hsn_sac_code", "hsn/sac", "hsn sac", "sac", "sac code"],
        "code_type": ["code_type", "code type"],
        "quantity": ["quantity", "qty", "nos", "pcs"],
        "unit": ["unit", "uom", "per", "unit of measure"],
        "unit_price": ["unit_price", "unit price", "rate", "unit rate", "price"],
        "discount": ["discount", "discount amount", "disc"],
        "taxable_amount": ["taxable_amount", "taxable amount", "taxable value", "assessable value"],
        "line_amount": ["line_amount", "line amount", "amount", "value"],
        "gst_rate": ["gst_rate", "gst rate", "gst %", "tax rate"],
        "cgst_rate": ["cgst_rate", "cgst rate", "cgst %"],
        "cgst_amount": ["cgst_amount", "cgst amount", "cgst"],
        "sgst_rate": ["sgst_rate", "sgst rate", "sgst %"],
        "sgst_amount": ["sgst_amount", "sgst amount", "sgst"],
        "igst_rate": ["igst_rate", "igst rate", "igst %", "integrated gst rate"],
        "igst_amount": ["igst_amount", "igst amount", "igst", "integrated gst"],
        "total": ["total", "line total", "item total", "total amount"],
    }

    lookup = {_norm_key(a): canon for canon, vals in aliases.items() for a in vals}
    numeric = {
        "quantity", "unit_price", "discount", "taxable_amount", "line_amount",
        "gst_rate", "cgst_rate", "cgst_amount", "sgst_rate", "sgst_amount",
        "igst_rate", "igst_amount", "total"
    }

    for k, v in item.items():
        canon = lookup.get(_norm_key(k))
        if not canon:
            if k not in ("raw_fields", "additional_fields"):
                extra[str(k)] = v
            continue
        out[canon] = _clean_numeric(v) if canon in numeric else v

    if not out.get("hsn_sac_code") and out.get("hsn_code"):
        out["hsn_sac_code"] = str(out["hsn_code"])

    if not out.get("code_type"):
        source_keys = " ".join(str(k) for k in item.keys()).lower()
        if "sac" in source_keys and "hsn" not in source_keys:
            out["code_type"] = "SAC"
        elif "hsn" in source_keys or out.get("hsn_code"):
            out["code_type"] = "HSN"

    # Explicit IGST/CGST/SGST is more specific than generic GST.
    if out.get("igst_rate") is not None or out.get("cgst_rate") is not None or out.get("sgst_rate") is not None:
        out["gst_rate"] = None

    # A value equal to explicit tax is not the line amount.
    explicit_tax = next(
        (out[k] for k in ("igst_amount", "cgst_amount", "sgst_amount") if out.get(k) is not None),
        None
    )
    if explicit_tax is not None and out.get("line_amount") == explicit_tax:
        out["line_amount"] = None

    if raw:
        out["raw_fields"] = raw
    if extra:
        out["additional_fields"] = extra

    return out


SUMMARY_PHRASES = [
    r"^\s*sub\s*total\b",
    r"^\s*subtotal\b",
    r"^\s*grand\s*total\b",
    r"^\s*total\s+in\s+",
    r"^\s*total\s+invoice\b",
    r"^\s*invoice\s+total\b",
    r"^\s*amount\s+payable\b",
    r"^\s*net\s+payable\b",
    r"^\s*taxable\s+value\b",
    r"^\s*integrated\s+gst\b",
    r"^\s*igst\b",
    r"^\s*cgst\b",
    r"^\s*sgst\b",
    r"^\s*gst\s+summary\b",
    r"^\s*tax\s+summary\b",
    r"^\s*total\s+tax\b",
    r"^\s*taxes?\b",
]


def _looks_like_summary_row_v5(item: dict) -> bool:
    desc = str(item.get("description") or "").strip().lower()

    for pattern in SUMMARY_PHRASES:
        if re.search(pattern, desc):
            return True

    if desc and re.fullmatch(
        r"(?:add\s*:\s*)?(?:integrated\s+gst|igst|cgst|sgst|gst|tax)"
        r"(?:\s*@\s*\d+(?:\.\d+)?\s*%?)?",
        desc
    ):
        return True

    return False


def _clean_tax_suffix_from_description_v5(item: dict) -> dict:
    desc = str(item.get("description") or "").strip()
    if not desc:
        return item

    m = re.search(
        r"\s+(?:add\s*:\s*)?(IGST|CGST|SGST|GST)\s*@\s*(\d+(?:\.\d+)?)\s*%?\s*$",
        desc,
        flags=re.I
    )
    if not m:
        return item

    prefix = desc[:m.start()].strip()
    if not prefix:
        return item

    tax_label = m.group(1).upper()
    rate = float(m.group(2))
    item["description"] = prefix

    if tax_label == "IGST" and item.get("igst_rate") is None:
        item["igst_rate"] = rate
    elif tax_label == "CGST" and item.get("cgst_rate") is None:
        item["cgst_rate"] = rate
    elif tax_label == "SGST" and item.get("sgst_rate") is None:
        item["sgst_rate"] = rate
    elif tax_label == "GST" and item.get("gst_rate") is None:
        item["gst_rate"] = rate

    if item.get("igst_rate") is not None:
        item["gst_rate"] = None

    return item


def _dedupe_and_filter_line_items_v5(items):
    result = []
    seen = set()

    for raw_item in items or []:
        li = _normalize_line_item_v5(raw_item)
        if not li:
            continue

        li = _clean_tax_suffix_from_description_v5(li)

        if _looks_like_summary_row_v5(li):
            continue

        has_business_content = any(
            li.get(k) not in (None, "")
            for k in (
                "description", "hsn_code", "hsn_sac_code", "quantity",
                "unit_price", "taxable_amount", "line_amount",
                "gst_rate", "cgst_amount", "sgst_amount",
                "igst_amount", "total"
            )
        )
        if not has_business_content:
            continue

        key = (
            str(li.get("description") or "").strip().lower(),
            str(li.get("hsn_sac_code") or li.get("hsn_code") or "").strip().lower(),
            li.get("quantity"),
            li.get("unit"),
            li.get("unit_price"),
            li.get("taxable_amount"),
            li.get("line_amount"),
            li.get("total"),
        )

        if key not in seen:
            seen.add(key)
            result.append(li)

    return result


def _merge_nonempty_v5(base: dict, incoming: dict, fields: list[str]):
    for field in fields:
        value = incoming.get(field)
        if value not in (None, ""):
            base[field] = value


def _explicit_source_labels(data: dict) -> set[str]:
    labels = set()

    def walk(obj):
        if isinstance(obj, dict):
            for k, v in obj.items():
                if isinstance(v, dict):
                    walk(v)
                else:
                    labels.add(_norm_key(str(k)))

    for source in (
        data.get("raw_fields") if isinstance(data.get("raw_fields"), dict) else {},
        data.get("additional_fields") if isinstance(data.get("additional_fields"), dict) else {},
    ):
        walk(source)

    return labels


def _repair_financial_semantics_v5(data: dict) -> dict:
    """
    Only retain a financial concept when there is source evidence for
    that concept. This prevents Taxable Value/Total from being silently
    relabeled as Subtotal.
    """
    out = dict(data or {})
    labels = _explicit_source_labels(out)

    def has_any(*names):
        return any(_norm_key(n) in labels for n in names)

    if out.get("subtotal") is not None and not has_any(
        "Subtotal", "Subtotal in INR", "Subtotal Amount"
    ):
        out["subtotal"] = None

    if out.get("taxable_amount") is not None and not has_any(
        "Taxable Value", "Taxable Amount", "Assessable Value", "Taxable in INR"
    ):
        out["taxable_amount"] = None

    if out.get("total_amount") is not None and not has_any(
        "Grand Total", "Total Amount", "Total Invoice Value",
        "Total in INR", "Net Payable", "Amount Payable", "Total"
    ):
        out["total_amount"] = None

    if out.get("igst_total") is not None and not has_any(
        "Total IGST", "IGST Total", "Integrated GST", "Integrated GST Amount"
    ):
        out["igst_total"] = None

    return out


def _extract_bank_identifiers_v5(data: dict) -> dict:
    """Safely parse bank labels, including combined Branch & IFS Code."""
    out = dict(data or {})
    bd = dict(out.get("bank_details") or {})
    raw = out.get("raw_fields") if isinstance(out.get("raw_fields"), dict) else {}
    extra = out.get("additional_fields") if isinstance(out.get("additional_fields"), dict) else {}

    sources = list(raw.items()) + list(extra.items())

    for key, value in sources:
        nk = _norm_key(str(key))
        txt = str(value).strip()

        if "a/c no" in nk or "account no" in nk or "account number" in nk or "bank a/c no" in nk:
            if not bd.get("account_number"):
                bd["account_number"] = txt

        if "branch" in nk and ("ifs" in nk or "ifsc" in nk):
            ifsc = re.search(r"\b[A-Z]{4}0[A-Z0-9]{6}\b", txt.upper())
            if ifsc:
                bd["ifsc_code"] = ifsc.group(0)

            branch_text = re.sub(
                r"\s*&?\s*(?:IFSC|IFS)\s*(?:CODE)?\s*:?\s*[A-Z]{4}0[A-Z0-9]{6}\b",
                "",
                txt,
                flags=re.I,
            ).strip(" &:-")
            if branch_text:
                bd["branch"] = branch_text

        if "bank name" in nk and not bd.get("bank_name"):
            bd["bank_name"] = txt

    out["bank_details"] = bd
    return out


def _merge_invoice_parts_v5(header: dict, footer_parts: list[dict], all_line_items: list[dict], raw_parts: dict):
    out = loosely_validate(header or {}) or InvoiceSchema().model_dump()

    financial_fields = [
        "subtotal", "taxable_amount", "discount_total", "tax_total",
        "cgst_total", "sgst_total", "igst_total", "shipping_charges",
        "other_charges", "round_off", "total_amount", "currency"
    ]

    for part in footer_parts:
        if not isinstance(part, dict):
            continue

        _merge_nonempty_v5(out, part, financial_fields)

        if isinstance(part.get("bank_details"), dict):
            bd = out.get("bank_details") if isinstance(out.get("bank_details"), dict) else {}
            for k, v in part["bank_details"].items():
                if v not in (None, ""):
                    bd[k] = v
            out["bank_details"] = bd

        if isinstance(part.get("additional_fields"), dict):
            out.setdefault("additional_fields", {}).update(part["additional_fields"])

        if isinstance(part.get("raw_fields"), dict):
            out.setdefault("raw_fields", {}).update(part["raw_fields"])

    out["line_items"] = _dedupe_and_filter_line_items_v5(all_line_items)
    out.setdefault("additional_fields", {})
    out.setdefault("raw_fields", {})
    out["raw_fields"]["page_passes"] = raw_parts

    out = _repair_financial_semantics_v5(out)
    out = _extract_bank_identifiers_v5(out)

    return loosely_validate(out) or out


def build_messages_header(image_path: str):
    return [{
        "role": "system",
        "content": [{"type": "text", "text": SYSTEM_PROMPT_HEADER}]
    }, {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": image_path,
                "min_pixels": MIN_PIXELS_FULL,
                "max_pixels": MAX_PIXELS_FULL,
            },
            {
                "type": "text",
                "text": (
                    "Populate every clearly visible document-level field. "
                    "Keep duplicate source labels in raw_fields. "
                    "Do not extract line-item rows. Return JSON only."
                ),
            },
        ],
    }]


def build_messages_line_items(crop_path: str):
    return [{
        "role": "system",
        "content": [{"type": "text", "text": SYSTEM_PROMPT_LINE_ITEMS}]
    }, {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": crop_path,
                "min_pixels": MIN_PIXELS_CROP,
                "max_pixels": MAX_PIXELS_CROP,
            },
            {
                "type": "text",
                "text": (
                    "First identify the actual billed-item/product/service table. "
                    "Extract ONLY genuine billed rows from that table. "
                    "Exclude subtotal, tax, GST/IGST, total, payable and summary rows. "
                    "Return [] when no genuine item row exists. Return JSON array only."
                ),
            },
        ],
    }]


def build_messages_footer(crop_path: str):
    return [{
        "role": "system",
        "content": [{"type": "text", "text": SYSTEM_PROMPT_FOOTER}]
    }, {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": crop_path,
                "min_pixels": MIN_PIXELS_CROP,
                "max_pixels": MAX_PIXELS_CROP,
            },
            {
                "type": "text",
                "text": (
                    "Extract only explicit invoice-level totals, taxes, charges, "
                    "bank/payment details and document summary fields. "
                    "Do not output line items. Return JSON object only."
                ),
            },
        ],
    }]


def _page_similarity(path_a: str, path_b: str) -> float:
    """
    Visual-only similarity using rendered pixels. No OCR or text recognition.
    """
    a = Image.open(path_a).convert("L").resize((300, 420))
    b = Image.open(path_b).convert("L").resize((300, 420))

    np = __import__("numpy")
    aa = np.asarray(a, dtype="float32")
    bb = np.asarray(b, dtype="float32")

    diff = np.mean(np.abs(aa - bb)) / 255.0
    return float(1.0 - diff)


def collapse_duplicate_invoice_pages(page_paths: list[str]):
    """
    Collapse near-identical Original/Duplicate/Triplicate pages.
    Keep genuine continuation pages.
    """
    if not ENABLE_DUPLICATE_PAGE_DETECTION or len(page_paths) <= 1:
        return page_paths, {
            "enabled": ENABLE_DUPLICATE_PAGE_DETECTION,
            "threshold": DUPLICATE_PAGE_SIMILARITY_THRESHOLD,
            "canonical_pages": list(range(1, len(page_paths) + 1)),
            "duplicate_groups": [],
        }

    canonical = []
    duplicate_groups = []
    used = [False] * len(page_paths)

    for i, path_i in enumerate(page_paths):
        if used[i]:
            continue

        used[i] = True
        canonical.append(path_i)
        group = [i + 1]

        for j in range(i + 1, len(page_paths)):
            if used[j]:
                continue

            sim = _page_similarity(path_i, page_paths[j])
            if sim >= DUPLICATE_PAGE_SIMILARITY_THRESHOLD:
                used[j] = True
                group.append(j + 1)

        if len(group) > 1:
            duplicate_groups.append({
                "canonical_page": group[0],
                "duplicate_pages": group[1:],
            })

    canonical_numbers = [
        page_paths.index(p) + 1
        for p in canonical
    ]

    return canonical, {
        "enabled": True,
        "threshold": DUPLICATE_PAGE_SIMILARITY_THRESHOLD,
        "canonical_pages": canonical_numbers,
        "duplicate_groups": duplicate_groups,
    }


def generate_invoice_document_v5(
    page_paths: list[str],
    header_max_new_tokens: int = 1400,
    line_items_max_new_tokens: int = 3200,
    footer_max_new_tokens: int = 1200,
):
    if not page_paths:
        return None, {"error": "no_pages"}

    canonical_paths, page_selection = collapse_duplicate_invoice_pages(page_paths)

    raw_parts = {
        "page_selection": page_selection,
        "header": None,
        "pages": []
    }

    # First canonical page establishes document identity.
    header_raw = _generate_raw(
        build_messages_header(canonical_paths[0]),
        header_max_new_tokens
    )
    header = _parse_json_object(header_raw)
    if header is None:
        header = _repair_unparseable_json_with_model(header_raw, "object")

    raw_parts["header"] = header_raw

    all_line_items = []
    footer_parts = []

    for logical_idx, page_path in enumerate(canonical_paths, start=1):
        page_entry = {
            "page": logical_idx,
            "source_path": page_path,
            "line_items_raw": None,
            "footer_raw": None,
            "line_items_parse_status": None,
            "footer_parse_status": None,
        }

        table_crop = save_vertical_crop(
            page_path,
            TABLE_TOP_RATIO,
            TABLE_BOTTOM_RATIO,
            f"p{logical_idx}_table"
        )

        line_raw = _generate_raw(
            build_messages_line_items(table_crop),
            line_items_max_new_tokens
        )
        line_parsed = _parse_json_array(line_raw)

        if line_parsed is None:
            line_parsed = _repair_unparseable_json_with_model(line_raw, "array")

        page_entry["line_items_raw"] = line_raw
        page_entry["line_items_parse_status"] = (
            "parsed" if line_parsed is not None else "unparseable"
        )

        if isinstance(line_parsed, list):
            all_line_items.extend(line_parsed)

        # Only the final canonical page supplies invoice-level summary/bank fields.
        if logical_idx == len(canonical_paths):
            footer_crop = save_vertical_crop(
                page_path,
                FOOTER_TOP_RATIO,
                FOOTER_BOTTOM_RATIO,
                f"p{logical_idx}_footer"
            )

            footer_raw = _generate_raw(
                build_messages_footer(footer_crop),
                footer_max_new_tokens
            )

            footer = _parse_json_object(footer_raw)
            if footer is None:
                footer = _repair_unparseable_json_with_model(footer_raw, "object")

            page_entry["footer_raw"] = footer_raw
            page_entry["footer_parse_status"] = (
                "parsed" if footer is not None else "unparseable"
            )

            if isinstance(footer, dict):
                footer_parts.append(footer)

        raw_parts["pages"].append(page_entry)

    if header is None:
        return None, raw_parts

    merged = _merge_invoice_parts_v5(
        header,
        footer_parts,
        all_line_items,
        raw_parts
    )

    return merged, raw_parts


def extract_document(page_paths: list[str]) -> dict:
    """Extraction-only entry point. Mathematical/business validation is external."""
    t0 = time.time()

    try:
        data, raw_texts = generate_invoice_document_v5(page_paths)
    except Exception as e:
        return {
            "data": None,
            "needs_review": True,
            "review_reasons": [f"generation_error:{type(e).__name__}: {e}"],
            "raw_output": None,
            "generation_path": "generation_error",
            "elapsed_seconds": round(time.time() - t0, 2),
        }

    if data is None:
        return {
            "data": None,
            "needs_review": True,
            "review_reasons": ["model_output_unparseable"],
            "raw_output": raw_texts,
            "generation_path": "unparseable",
            "elapsed_seconds": round(time.time() - t0, 2),
        }

    data, resolved_sources, _ = resolve_invoice_fields(data)

    return {
        "data": data,
        "field_sources": {
            **{f: "llm" for f in CORE_FIELDS if data.get(f) is not None},
            **resolved_sources
        },
        "needs_review": False,
        "review_reasons": [],
        "raw_output": raw_texts,
        "generation_path": "qwen3vl_v5_extraction_only",
        "elapsed_seconds": round(time.time() - t0, 2),
    }

## 7. What changed from the previous notebook

### Visual processing
- Qwen3-VL-native preprocessing: `image_patch_size=16`
- `do_resize=False` to avoid a second resize after `process_vision_info`
- 32×32 effective visual-unit math instead of the old 28×28 Qwen2.5-style assumption
- Larger visual budget for invoice crops

### Extraction strategy
- Full page → identity/header
- High-resolution overlapping body crop → line items
- High-resolution lower-page crop → totals, taxes, bank/payment fields
- No external OCR engine
- JSON is retained, but each model call has a smaller job

### Model testing
- One switch between Qwen3-VL-4B-Instruct and Qwen3-VL-8B-Instruct
- 4-bit remains available for Colab/local testing
- Production validation can run with `USE_4BIT=False`

The goal is to test whether the current accuracy problem is caused primarily by visual information loss and task overload before moving to a bigger model.

In [ ]:
_FENCE_RE = re.compile(r"^```(?:json)?\s*|\s*```$", re.MULTILINE)

def strip_code_fences(text):
    return _FENCE_RE.sub("", text or "").strip()

def extract_json_block(text):
    start = text.find("{")
    if start == -1: return None
    depth = 0; in_string = False; escape = False
    for i in range(start, len(text)):
        ch = text[i]
        if in_string:
            if escape: escape = False
            elif ch == "\\": escape = True
            elif ch == '"': in_string = False
            continue
        if ch == '"': in_string = True
        elif ch == "{": depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0: return text[start:i+1]
    return None

def _extract_json_array(text):
    cleaned = strip_code_fences(text)
    start = cleaned.find("[")
    if start == -1: return None
    depth = 0; in_string = False; escape = False
    for i in range(start, len(cleaned)):
        ch = cleaned[i]
        if in_string:
            if escape: escape = False
            elif ch == "\\": escape = True
            elif ch == '"': in_string = False
            continue
        if ch == '"': in_string = True
        elif ch == "[": depth += 1
        elif ch == "]":
            depth -= 1
            if depth == 0: return cleaned[start:i+1]
    return None

def repair_truncated_json(text):
    s = (text or "").strip()
    if not s: return None
    start = s.find("{")
    if start == -1: return None
    s = s[start:]
    stack=[]; in_string=False; escape=False
    for ch in s:
        if in_string:
            if escape: escape=False
            elif ch == "\\": escape=True
            elif ch == '"': in_string=False
        else:
            if ch == '"': in_string=True
            elif ch in "{[": stack.append(ch)
            elif ch in "}]" and stack: stack.pop()
    if in_string:
        s=s[:max(s.rfind(','), s.rfind('{'))]
        stack=[]; in_string=False; escape=False
        for ch in s:
            if in_string:
                if escape: escape=False
                elif ch == "\\": escape=True
                elif ch == '"': in_string=False
            else:
                if ch == '"': in_string=True
                elif ch in "{[": stack.append(ch)
                elif ch in "}]" and stack: stack.pop()
    s=re.sub(r",\s*$", "", s)
    s += "".join({"{":"}", "[":"]"}[x] for x in reversed(stack))
    return s

def safe_json_parse(raw_text):
    cleaned = strip_code_fences(raw_text)
    candidates = [cleaned, extract_json_block(cleaned)]
    for candidate in candidates:
        if candidate:
            try: return json.loads(candidate)
            except json.JSONDecodeError: pass
    repaired=repair_truncated_json(cleaned)
    if repaired:
        try: return json.loads(repaired)
        except json.JSONDecodeError: pass
    return None

NUMERIC_TOP_FIELDS = {"subtotal","taxable_amount","discount_total","tax_total","cgst_total","sgst_total","igst_rate","igst_total","adjustment","shipping_charges","other_charges","round_off","total_amount"}
NUMERIC_LINE_ITEM_FIELDS = {"quantity","unit_price","discount","taxable_amount","line_amount","gst_rate","cgst_rate","cgst_amount","sgst_rate","sgst_amount","igst_rate","igst_amount","total"}

def _clean_numeric(v):
    if v is None: return None
    if isinstance(v,(int,float)): return float(v)
    if isinstance(v,str):
        s=v.strip(); negative=s.startswith("(") and s.endswith(")")
        s=re.sub(r"/[-=]\s*$","",s)
        s=re.sub(r"[A-Za-z₹$€£]+\.?","",s)
        cleaned=re.sub(r"[^\d.\-]","",s)
        if cleaned.count('.')>1:
            head,_,tail=cleaned.rpartition('.'); cleaned=head.replace('.','')+'.'+tail
        if cleaned in ("","-",".","-."): return None
        try: val=float(cleaned)
        except ValueError: return None
        return -abs(val) if negative else val
    return None

def _norm_key(k): return re.sub(r"[^a-z0-9]+"," ",str(k).lower()).strip()

def loosely_validate(data):
    if not isinstance(data,dict): return None
    working=dict(data)
    if isinstance(working.get('line_items'),list):
        working['line_items']=[_normalise_line_item_v3(li) for li in working['line_items'] if isinstance(li,dict)]
    for f in NUMERIC_TOP_FIELDS:
        if f in working: working[f]=_clean_numeric(working[f])
    filtered={k:v for k,v in working.items() if k in InvoiceSchema.model_fields}
    stray={k:v for k,v in working.items() if k not in InvoiceSchema.model_fields}
    if stray:
        filtered.setdefault('additional_fields',{}).update(stray)
    try:
        return InvoiceSchema(**filtered).model_dump()
    except ValidationError:
        # Keep all readable data even if a nested validator rejects one part.
        return filtered


def loosely_validate_header(data):
    return loosely_validate(data)

In [ ]:
_FENCE_RE = re.compile(r"^```(?:json)?\s*|\s*```$", re.MULTILINE)

def strip_code_fences(text):
    return _FENCE_RE.sub("", text or "").strip()


def extract_json_block(text):
    start = text.find("{")
    if start == -1:
        return None

    depth = 0
    in_string = False
    escape = False

    for i, ch in enumerate(text[start:], start=start):
        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue

        if ch == '"':
            in_string = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]

    return None


def _extract_json_array(text):
    cleaned = strip_code_fences(text)
    start = cleaned.find("[")
    if start == -1:
        return None

    depth = 0
    in_string = False
    escape = False

    for i, ch in enumerate(cleaned[start:], start=start):
        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
            continue

        if ch == '"':
            in_string = True
        elif ch == "[":
            depth += 1
        elif ch == "]":
            depth -= 1
            if depth == 0:
                return cleaned[start:i + 1]

    return None


def repair_truncated_json(text):
    s = (text or "").strip()
    if not s:
        return None

    start = s.find("{")
    if start == -1:
        start = s.find("[")
    if start == -1:
        return None

    s = s[start:]
    stack = []
    in_string = False
    escape = False

    for ch in s:
        if in_string:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_string = False
        else:
            if ch == '"':
                in_string = True
            elif ch in "{[":
                stack.append(ch)
            elif ch in "}]":
                if stack:
                    stack.pop()

    if in_string:
        trim_at = max(s.rfind(","), s.rfind("{"), s.rfind("["))
        s = s[:trim_at] if trim_at >= 0 else s

        stack = []
        in_string = False
        escape = False
        for ch in s:
            if in_string:
                if escape:
                    escape = False
                elif ch == "\\":
                    escape = True
                elif ch == '"':
                    in_string = False
            else:
                if ch == '"':
                    in_string = True
                elif ch in "{[":
                    stack.append(ch)
                elif ch in "}]":
                    if stack:
                        stack.pop()

    s = re.sub(r",\s*$", "", s)
    closers = {"{": "}", "[": "]"}
    s += "".join(closers[c] for c in reversed(stack))
    return s


def safe_json_parse(raw_text):
    cleaned = strip_code_fences(raw_text)

    candidates = [
        cleaned,
        extract_json_block(cleaned),
        _extract_json_array(cleaned),
    ]

    for candidate in candidates:
        if candidate:
            try:
                return json.loads(candidate)
            except json.JSONDecodeError:
                pass

    repaired = repair_truncated_json(cleaned)
    if repaired:
        try:
            return json.loads(repaired)
        except json.JSONDecodeError:
            pass

    return None


NUMERIC_TOP_FIELDS = {
    "subtotal", "taxable_amount", "discount_total", "tax_total",
    "cgst_total", "sgst_total", "igst_rate", "igst_total", "adjustment", "shipping_charges",
    "other_charges", "round_off", "total_amount"
}

NUMERIC_LINE_ITEM_FIELDS = {
    "quantity", "unit_price", "discount", "taxable_amount",
    "line_amount", "gst_rate", "cgst_rate", "cgst_amount",
    "sgst_rate", "sgst_amount", "igst_rate", "igst_amount", "total"
}


def _clean_numeric(v):
    if v is None:
        return None

    if isinstance(v, (int, float)):
        return float(v)

    if isinstance(v, str):
        s = v.strip()
        negative = s.startswith("(") and s.endswith(")")
        s = re.sub(r"/[-=]\s*$", "", s)
        s = re.sub(r"[A-Za-z₹$€£]+\.?", "", s)
        cleaned = re.sub(r"[^\d.\-]", "", s)

        if cleaned.count(".") > 1:
            head, _, tail = cleaned.rpartition(".")
            cleaned = head.replace(".", "") + "." + tail

        if cleaned in ("", "-", ".", "-."):
            return None

        try:
            val = float(cleaned)
        except ValueError:
            return None

        return -abs(val) if negative else val

    return None


def _norm_key(k):
    return re.sub(r"[^a-z0-9]+", " ", str(k).lower()).strip()


# IMPORTANT:
# This shared normalizer is used by loosely_validate() and therefore
# MUST understand every V5 canonical line-item field.
LINE_ITEM_KEY_ALIASES = {
    "description": {
        "description", "product name", "product name description", "item",
        "item description", "particulars", "product", "service", "goods"
    },
    "hsn_code": {
        "hsn code", "hsn"
    },
    "hsn_sac_code": {
        "hsn sac", "hsn/sac", "hsn sac code", "hsn_sac_code",
        "sac", "sac code"
    },
    "code_type": {
        "code type", "code_type"
    },
    "quantity": {
        "qty", "qty nos", "quantity", "nos", "pcs"
    },
    "unit": {
        "unit", "uom", "per", "unit of measure"
    },
    "unit_price": {
        "unit price", "rate", "price", "unit rate"
    },
    "discount": {
        "disc amt", "discount amt", "discount amount", "disc value", "discount"
    },
    "taxable_amount": {
        "taxable amt rs", "taxable amount", "taxable amt",
        "taxable value", "assessable value"
    },
    "line_amount": {
        "line amount", "line_amount", "amount", "value",
        "amount rs", "line value"
    },
    "gst_rate": {
        "gst rate", "gst %", "gst pct", "gst percent", "tax rate", "tax %"
    },
    "cgst_rate": {
        "cgst rate", "cgst %", "cgst pct", "cgst percent"
    },
    "cgst_amount": {
        "cgst rs", "cgst", "cgst amt", "cgst amount"
    },
    "sgst_rate": {
        "sgst rate", "sgst %", "sgst pct", "sgst percent"
    },
    "sgst_amount": {
        "sgst rs", "sgst", "sgst amt", "sgst amount"
    },
    "igst_rate": {
        "igst rate", "igst %", "igst pct", "igst percent", "integrated gst rate"
    },
    "igst_amount": {
        "igst rs", "igst", "igst amt", "igst amount", "integrated gst"
    },
    "total": {
        "total rs", "total", "amount total", "line total",
        "item total", "total amount"
    },
}

_ALIAS_LOOKUP = {
    _norm_key(alias): canonical
    for canonical, aliases in LINE_ITEM_KEY_ALIASES.items()
    for alias in aliases
}


def normalize_line_item(item):
    if not isinstance(item, dict):
        return {}

    out = {}
    stray = {}
    raw_fields = {}

    for k, v in item.items():
        canon = _ALIAS_LOOKUP.get(_norm_key(k))

        if canon is None:
            if str(k) not in ("additional_fields", "raw_fields"):
                stray[str(k)] = v
            continue

        if (
            isinstance(item.get("raw_fields"), dict)
            and canon in item["raw_fields"]
        ):
            raw_fields[canon] = item["raw_fields"][canon]
        elif v is not None and canon in NUMERIC_LINE_ITEM_FIELDS:
            raw_fields[canon] = v

        out[canon] = (
            _clean_numeric(v)
            if canon in NUMERIC_LINE_ITEM_FIELDS
            else v
        )

    # Preserve original raw_fields alongside normalized values.
    if isinstance(item.get("raw_fields"), dict):
        raw_fields = {**item["raw_fields"], **raw_fields}

    # Promote HSN -> hsn_sac_code when appropriate. Keep hsn_code for
    # backward compatibility, but don't delete the broader field.
    if not out.get("hsn_sac_code") and out.get("hsn_code"):
        out["hsn_sac_code"] = str(out["hsn_code"])

    if not out.get("code_type"):
        source_keys = " ".join(str(k) for k in item.keys()).lower()
        if "sac" in source_keys and "hsn" not in source_keys:
            out["code_type"] = "SAC"
        elif "hsn" in source_keys or out.get("hsn_code"):
            out["code_type"] = "HSN"

    # Explicit component tax rates are more specific than generic GST.
    if (
        out.get("igst_rate") is not None
        or out.get("cgst_rate") is not None
        or out.get("sgst_rate") is not None
    ):
        out["gst_rate"] = None

    # If line_amount accidentally equals explicit tax, don't label the
    # tax amount as the line's merchandise/service amount.
    explicit_tax = next(
        (
            out[k] for k in
            ("igst_amount", "cgst_amount", "sgst_amount")
            if out.get(k) is not None
        ),
        None
    )
    if (
        explicit_tax is not None
        and out.get("line_amount") is not None
        and out["line_amount"] == explicit_tax
    ):
        out["line_amount"] = None

    if raw_fields:
        out["raw_fields"] = raw_fields

    existing_extra = item.get("additional_fields")
    if isinstance(existing_extra, dict):
        stray = {**existing_extra, **stray}

    if stray:
        out["additional_fields"] = stray

    return out


def loosely_validate(data):
    """Normalize model JSON without running business/math validation."""
    if not isinstance(data, dict):
        return None

    working = dict(data)

    if isinstance(working.get("line_items"), list):
        working["line_items"] = [
            normalize_line_item(li)
            for li in working["line_items"]
            if isinstance(li, dict)
        ]

    for field in NUMERIC_TOP_FIELDS:
        if field in working:
            working[field] = _clean_numeric(working[field])

    filtered = {
        k: v for k, v in working.items()
        if k in InvoiceSchema.model_fields
    }

    stray = {
        k: v for k, v in working.items()
        if k not in InvoiceSchema.model_fields
    }

    if stray:
        filtered.setdefault("additional_fields", {})
        if isinstance(filtered["additional_fields"], dict):
            filtered["additional_fields"].update(stray)

    for _ in range(len(InvoiceSchema.model_fields) + 2):
        try:
            return InvoiceSchema(**filtered).model_dump()
        except ValidationError as e:
            bad_fields = {
                err["loc"][0]
                for err in e.errors()
                if err.get("loc")
            }

            if not bad_fields:
                return None

            progressed = False

            for bad in bad_fields:
                if bad == "line_items":
                    kept = []

                    for li in filtered.get("line_items", []):
                        try:
                            kept.append(LineItem(**li).model_dump())
                        except ValidationError:
                            pass

                    if len(kept) != len(filtered.get("line_items", [])):
                        filtered["line_items"] = kept
                        progressed = True

                elif bad in filtered:
                    value = filtered.pop(bad)
                    filtered.setdefault("additional_fields", {})
                    if isinstance(filtered["additional_fields"], dict):
                        filtered["additional_fields"][f"unparsed_{bad}"] = value
                    progressed = True

            if not progressed:
                return None

    return None


def loosely_validate_header(data):
    if not isinstance(data, dict):
        return None

    working = dict(data)

    for field in NUMERIC_TOP_FIELDS:
        if field in working:
            working[field] = _clean_numeric(working[field])

    filtered = {
        k: v for k, v in working.items()
        if k in InvoiceHeaderSchema.model_fields
    }

    stray = {
        k: v for k, v in working.items()
        if k not in InvoiceHeaderSchema.model_fields
    }

    if stray:
        filtered.setdefault("additional_fields", {})
        if isinstance(filtered["additional_fields"], dict):
            filtered["additional_fields"].update(stray)

    for _ in range(len(InvoiceHeaderSchema.model_fields) + 2):
        try:
            return InvoiceHeaderSchema(**filtered).model_dump()
        except ValidationError as e:
            bad_fields = {
                err["loc"][0]
                for err in e.errors()
                if err.get("loc")
            }

            if not bad_fields:
                return None

            progressed = False
            for bad in bad_fields:
                if bad in filtered:
                    value = filtered.pop(bad)
                    filtered.setdefault("additional_fields", {})
                    if isinstance(filtered["additional_fields"], dict):
                        filtered["additional_fields"][f"unparsed_{bad}"] = value
                    progressed = True

            if not progressed:
                return None

    return None

## 10. Identifier validation and VLM re-read fallback

In [ ]:
from datetime import timedelta

CODE36 = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"

PATTERNS = {
    "vendor_gstin": re.compile(r"\b\d{2}[A-Z]{5}\d{4}[A-Z][1-9A-Z]Z[0-9A-Z]\b"),
    "customer_gstin": re.compile(r"\b\d{2}[A-Z]{5}\d{4}[A-Z][1-9A-Z]Z[0-9A-Z]\b"),
    "vendor_pan": re.compile(r"\b[A-Z]{5}\d{4}[A-Z]\b"),
    "customer_pan": re.compile(r"\b[A-Z]{5}\d{4}[A-Z]\b"),
    "vendor_cin": re.compile(r"\b[UL]\d{5}[A-Z]{2}\d{4}[A-Z]{3}\d{6}\b"),
    "vendor_phone": re.compile(r"\b(?:\+?91[\-\s]?)?[6-9]\d{9}\b"),
    "vendor_email": re.compile(r"[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}"),
    "invoice_date": re.compile(r"\b\d{1,2}[\/\-\.]\d{1,2}[\/\-\.]\d{2,4}\b|\b\d{4}[\/\-]\d{1,2}[\/\-]\d{1,2}\b"),
    "due_date": re.compile(r"\b\d{1,2}[\/\-\.]\d{1,2}[\/\-\.]\d{2,4}\b|\b\d{4}[\/\-]\d{1,2}[\/\-]\d{1,2}\b"),
    # Bank sub-fields -- only used to fill a gap the model left null, never
    # to override an extracted value (see fill_missing_bank_fields).
    "ifsc_code": re.compile(r"\b[A-Z]{4}0[A-Z0-9]{6}\b"),
    "account_number": re.compile(r"\b\d{9,18}\b"),
    "upi_id": re.compile(r"\b[a-zA-Z0-9.\-_]{2,64}@[a-zA-Z]{2,64}\b"),
}

LABEL_HINTS = {
    "vendor_gstin": ["gstin", "gst no", "gst reg", "gst number", "gst in", "tax registration"],
    "customer_gstin": ["gstin", "gst no", "gst reg", "gst number"],
    "vendor_pan": ["pan no", "pan number", "pan:", "permanent account number"],
    "customer_pan": ["pan no", "pan number", "pan:", "permanent account number"],
    "vendor_cin": ["cin", "cin no", "corporate identity"],
    "vendor_phone": ["phone", "mobile", "contact", "tel", "ph."],
    "vendor_email": ["email", "e-mail", "mail id"],
    "invoice_date": ["invoice date", "date", "dated", "bill date", "document date", "issue date"],
    "due_date": ["due date", "payment due", "payable by", "pay by"],
    "ifsc_code": ["ifsc", "ifsc code", "ifs code"],
    "account_number": ["a/c no", "account no", "account number", "bank a/c", "acc no", "a c no"],
    "upi_id": ["upi", "upi id", "vpa"],
}

def is_valid_gstin_format(v):
    return bool(v) and bool(PATTERNS["vendor_gstin"].fullmatch(v.strip().upper()))

def is_valid_gstin_checksum(v):
    # Verified against two independently-published GSTIN Luhn mod-36 worked
    # examples (27AABCU9603R1ZN and 27AAPFU0939F1ZV) -- this formula is correct
    # as originally written. Left unchanged; do not "fix" this again without
    # re-verifying against a real worked example first.
    if not v or len(v) != 15:
        return False
    v = v.strip().upper()
    try:
        factor = 1
        total = 0
        for ch in v[:-1]:
            digit = CODE36.index(ch)
            code_point = factor * digit
            factor = 2 if factor == 1 else 1
            code_point = (code_point // 36) + (code_point % 36)
            total += code_point
        check_digit = CODE36[(36 - (total % 36)) % 36]
        return check_digit == v[-1]
    except (ValueError, IndexError):
        return False

def is_valid_pan_format(v):
    return bool(v) and bool(re.fullmatch(r"[A-Z]{5}\d{4}[A-Z]", v.strip().upper()))

def is_valid_ifsc_format(v):
    return bool(v) and bool(re.fullmatch(r"[A-Z]{4}0[A-Z0-9]{6}", v.strip().upper()))

# Only ifsc_code/account_number/upi_id have patterns reliable enough to
# regex-match on their own; bank_name/branch/account_holder_name are left
# exactly as the model returned them (see fill_missing_bank_fields).
BANK_FIELD_VALIDATORS = {
    "ifsc_code": is_valid_ifsc_format,
    "account_number": lambda v: bool(v) and v.strip().isdigit() and 9 <= len(v.strip()) <= 18,
    "upi_id": lambda v: bool(v) and bool(PATTERNS["upi_id"].fullmatch(v.strip())),
}

def derive_pan_from_gstin(gstin):
    if gstin and len(gstin) == 15:
        candidate = gstin[2:12].upper()
        if is_valid_pan_format(candidate):
            return candidate
    return None

def normalize_date(value):
    if not value:
        return None
    try:
        dt = dateparser.parse(value, dayfirst=True, fuzzy=True)
        return dt.strftime("%Y-%m-%d")
    except (ValueError, OverflowError, TypeError):
        return value

# "Net 30", "30 days from invoice date", "Due on receipt", "50% advance", etc.
# Deliberately simple -- anything it can't parse just leaves due_date null,
# same as before; it never guesses.
_NET_TERMS_RE = re.compile(r"net\s*(\d{1,3})|(\d{1,3})\s*days?", re.IGNORECASE)
_IMMEDIATE_TERMS = ("due on receipt", "immediate", "advance", "cash", "cod", "payable on receipt", "prepaid")

def compute_due_date_from_terms(payment_terms, invoice_date):
    """Best-effort due date from payment_terms text + invoice_date. Only ever
    used to FILL a missing due_date -- never overrides a due_date the model
    actually read off the invoice (see extract_invoice).

    NOTE: by the time this runs, invoice_date has already been through
    normalize_date() and is YYYY-MM-DD -- parse it WITHOUT dayfirst=True, or
    an unambiguous ISO date like 2026-07-01 gets misread as day=07 (dayfirst
    forces day-month-year even on year-first strings)."""
    if not payment_terms or not invoice_date:
        return None
    try:
        base = dateparser.parse(invoice_date)
    except (ValueError, OverflowError, TypeError):
        return None
    terms_lower = payment_terms.lower()
    if any(term in terms_lower for term in _IMMEDIATE_TERMS):
        return base.strftime("%Y-%m-%d")
    m = _NET_TERMS_RE.search(terms_lower)
    if m:
        days = int(m.group(1) or m.group(2))
        return (base + timedelta(days=days)).strftime("%Y-%m-%d")
    return None

def regex_fallback(field, raw_text, start_after=0):
    """Returns (matched_value_or_None, match_start_index_or_-1).

    Accepts `start_after` so a caller can force a second, related lookup
    (e.g. customer_gstin after vendor_gstin) to search past the first match
    instead of re-finding the same label/window -- otherwise vendor_gstin and
    customer_gstin can both resolve to the same text since they share label
    keywords like "gstin".
    """
    pattern = PATTERNS.get(field)
    if pattern is None or not raw_text:
        return None, -1
    lowered = raw_text.lower()
    for hint in LABEL_HINTS.get(field, []):
        idx = lowered.find(hint, start_after)
        if idx != -1:
            window = raw_text[max(0, idx - 10): idx + 120]
            m = pattern.search(window)
            if m:
                return m.group().strip(), idx
    m = pattern.search(raw_text, start_after)
    return (m.group().strip(), m.start()) if m else (None, -1)

TRANSCRIBE_PROMPT = (
    "Transcribe every piece of text visible in this image exactly as printed, "
    "line by line, including all labels, numbers, codes, and dates. Do not "
    "summarize, interpret, or omit anything. Plain text only -- no JSON, no "
    "commentary."
)

def _build_transcribe_messages(image_path):
    return [
        {"role": "system", "content": "You transcribe text from images verbatim."},
        {"role": "user", "content": [
            {"type": "image", "image": image_path, "min_pixels": MIN_PIXELS, "max_pixels": MAX_PIXELS},
            {"type": "text", "text": TRANSCRIBE_PROMPT},
        ]},
    ]

def get_backup_text_via_reread(image_path):
    """Re-reads the invoice image with the loaded VLM, asking for a raw
    transcription. Used as the fallback text source for regex_fallback()
    wherever a field fails validation. Wrapped in try/except because this is
    already a fallback path -- a second failure here should degrade to an
    empty string, not crash the pipeline."""
    try:
        return _run_generation(image_path, _build_transcribe_messages, max_new_tokens=1024)
    except Exception:
        return ""

## 10. Invoice arithmetic checks

In [ ]:
def reconcile_line_item(item, tolerance=0.02):
    qty, price = item.get("quantity"), item.get("unit_price")
    stated_total = item.get("total")

    if qty is None or price is None or stated_total is None:
        return {"status": "incomplete"}

    gross = qty * price
    discount_amt = item.get("discount") or 0
    taxable_declared = item.get("taxable_amount")
    cgst = item.get("cgst_amount")
    sgst = item.get("sgst_amount")
    igst = item.get("igst_amount")

    taxable_computed = gross - discount_amt
    base = taxable_declared if taxable_declared is not None else taxable_computed

    # Per-tax-type rates (cgst_rate/sgst_rate/igst_rate), summed -- an extra
    # arithmetic-check candidate alongside the amount-based one below.
    rate_parts = [item.get("cgst_rate"), item.get("sgst_rate"), item.get("igst_rate")]
    split_rate = sum(r for r in rate_parts if r is not None) if any(r is not None for r in rate_parts) else None

    has_tax_info = any(v is not None for v in (cgst, sgst, igst)) or split_rate is not None

    if not has_tax_info:
        if abs(gross - stated_total) <= tolerance:
            return {"status": "ok", "matched_formula": "gross_equals_total_no_tax"}
        if taxable_declared is not None and abs(taxable_declared - stated_total) <= tolerance:
            return {"status": "ok", "matched_formula": "taxable_equals_total_no_tax"}
        if stated_total > base + tolerance:
            return {
                "status": "unverifiable_tax_breakdown",
                "pretax_amount": round(base, 2), "stated_total": stated_total,
                "implied_tax": round(stated_total - base, 2),
            }
        return {"status": "mismatch", "candidates": {"gross_only": round(gross, 2)}, "stated": stated_total}

    tax_total = sum(v for v in (cgst, sgst, igst) if v is not None)

    candidates = {"taxable_plus_tax": base + tax_total}
    if split_rate is not None:
        candidates["taxable_times_split_rates"] = base * (1 + split_rate / 100)

    for label, val in candidates.items():
        if abs(val - stated_total) <= tolerance:
            return {"status": "ok", "matched_formula": label, "computed": round(val, 2)}

    return {"status": "mismatch", "candidates": {k: round(v, 2) for k, v in candidates.items()}, "stated": stated_total}


def reconcile_invoice(data, tolerance=0.5):
    line_items = data.get("line_items") or []
    line_total_sum = sum(li.get("total") for li in line_items if isinstance(li.get("total"), (int, float)))

    # Only sum taxable_amount when EVERY line item has it -- falling back to
    # `total` (post-tax) per-item when taxable_amount is missing would mix
    # pre-tax and post-tax bases into one "taxable" sum, which isn't
    # apples-to-apples.
    taxable_vals = [li.get("taxable_amount") for li in line_items if isinstance(li.get("taxable_amount"), (int, float))]
    taxable_basis_complete = len(line_items) > 0 and len(taxable_vals) == len(line_items)
    line_taxable_sum = sum(taxable_vals) if taxable_vals else None

    subtotal = data.get("subtotal")
    discount_total = data.get("discount_total") or 0
    tax_total = data.get("tax_total") or 0
    shipping = data.get("shipping_charges") or 0
    other_charges = data.get("other_charges") or 0
    round_off = data.get("round_off") or 0
    total_amount = data.get("total_amount")

    result = {"line_items_sum": round(line_total_sum, 2) if line_items else None}

    if subtotal is not None and line_items:
        if taxable_basis_complete:
            result["subtotal_match"] = abs(line_taxable_sum - subtotal) <= tolerance
        else:
            result["subtotal_match"] = None
            result["subtotal_check_note"] = "incomplete_taxable_amount_on_line_items"

    if total_amount is not None:
        # subtotal is defined as the post-discount taxable value in the
        # prompt ("Taxable Amount"/"Taxable Value"/"Assessable Value"), so it
        # is already net of discount and discount_total is NOT subtracted a
        # second time here.
        base = subtotal if subtotal is not None else (line_taxable_sum if taxable_basis_complete else line_total_sum)
        reconstructed = base + tax_total + shipping + other_charges + round_off
        result["total_reconstructed"] = round(reconstructed, 2)
        result["total_match"] = abs(reconstructed - total_amount) <= tolerance
        if discount_total:
            result["total_reconciliation_note"] = (
                "discount_total not subtracted here -- subtotal is assumed already "
                "net of discount per the extraction prompt's field definitions"
            )

    return result

## 10. Lossless tax metadata helper

In [ ]:
def ensure_lossless_tax_metadata(invoice: dict) -> dict:
    """
    Add a structured tax_details layer while preserving the raw VLM
    additional_fields exactly. This is additive and lossless.
    """
    if not isinstance(invoice, dict):
        return invoice

    extras = invoice.get("additional_fields")
    if not isinstance(extras, dict):
        extras = {}
        invoice["additional_fields"] = extras

    tax_details = extras.get("tax_details")
    if not isinstance(tax_details, dict):
        tax_details = {}
        extras["tax_details"] = tax_details

    # Preserve any tax_details generated by the VLM. Also surface common
    # top-level/extra labels into structured semantic buckets without
    # deleting the originals.
    semantic_map = {
        "cgst": ("output_tax", "cgst"),
        "sgst": ("output_tax", "sgst"),
        "igst": ("output_tax", "igst"),
        "cess": ("output_tax", "cess"),
        "input cgst": ("input_tax_credit", "input_cgst"),
        "input sgst": ("input_tax_credit", "input_sgst"),
        "input igst": ("input_tax_credit", "input_igst"),
        "input cess": ("input_tax_credit", "input_cess"),
        "tds": ("tds", "amount"),
        "tcs": ("tcs", "amount"),
        "tax payable": ("tax_payable", "amount"),
        "gst payable": ("tax_payable", "amount"),
        "net tax payable": ("tax_payable", "amount"),
        "tax liability": ("tax_payable", "amount"),
    }

    def norm_label(label):
        return re.sub(r"[^a-z0-9]+", " ", str(label).lower()).strip()

    # Canonical line-level output tax information.
    output_tax = tax_details.setdefault("output_tax", {})
    for key, rate_key, amount_key in (
        ("cgst", "cgst_rate", "cgst_amount"),
        ("sgst", "sgst_rate", "sgst_amount"),
        ("igst", "igst_rate", "igst_amount"),
    ):
        rate = invoice.get(rate_key)
        amount = invoice.get(amount_key)
        if rate is not None or amount is not None:
            node = output_tax.setdefault(key, {})
            if rate is not None:
                node["rate"] = rate
            if amount is not None:
                node["amount"] = amount

    # Existing raw keys in additional_fields.
    for raw_key, raw_value in list(extras.items()):
        if raw_key == "tax_details":
            continue
        k = norm_label(raw_key)
        if k not in semantic_map:
            continue
        bucket, leaf = semantic_map[k]
        bucket_obj = tax_details.setdefault(bucket, {})
        if bucket == "tds":
            bucket_obj.setdefault("amount", raw_value)
        elif bucket == "tcs":
            bucket_obj.setdefault("amount", raw_value)
        elif bucket == "tax_payable":
            bucket_obj.setdefault("amount", raw_value)
        else:
            node = bucket_obj.setdefault(leaf, {})
            node.setdefault("amount", raw_value)

    # Always make these buckets available so downstream LLM logic sees a
    # stable semantic structure. Nulls mean "not observed", not zero.
    tax_details.setdefault("output_tax", {})
    tax_details.setdefault("input_tax_credit", {})
    tax_details.setdefault("tds", {})
    tax_details.setdefault("tcs", {})
    tax_details.setdefault("tax_payable", {})
    tax_details.setdefault("other_tax_information", [])

    return invoice

## 11. Canonical-field resolver — extraction preservation

This resolver maps explicit source labels into canonical fields. **It does not run mathematical/business validation.**

In [ ]:
def _flat_raw_candidates(obj, prefix=""):
    out = []
    if not isinstance(obj, dict):
        return out

    for k, v in obj.items():
        if isinstance(v, dict):
            out.extend(_flat_raw_candidates(v, f"{prefix}.{k}" if prefix else str(k)))
        elif isinstance(v, (str, int, float)) and v not in (None, ""):
            out.append((_norm_key(k), str(k), v, prefix))

    return out


def _candidate_value(candidates, aliases):
    wanted = {_norm_key(a) for a in aliases}
    for nk, label, value, prefix in candidates:
        if nk in wanted:
            return label, value, prefix
    return None


def _resolve_amount_candidate(hit):
    return _clean_numeric(hit[1]) if hit else None


def _extract_ifsc_from_text(text: str):
    m = re.search(r"\b[A-Z]{4}0[A-Z0-9]{6}\b", str(text).upper())
    return m.group(0) if m else None


def resolve_invoice_fields(data):
    if not isinstance(data, dict):
        return data, {}, []

    out = copy.deepcopy(data)
    sources = {}
    reasons = []

    raw = out.get("raw_fields") if isinstance(out.get("raw_fields"), dict) else {}
    extra = out.get("additional_fields") if isinstance(out.get("additional_fields"), dict) else {}
    candidates = _flat_raw_candidates(raw) + _flat_raw_candidates(extra)

    # Raw source-label resolution is a fallback, never a destructive validation step.
    text_mappings = {
        "invoice_number": ["Invoice No.", "Invoice Number", "Inv No.", "Inv. No.", "Bill No.", "Bill Number"],
        "invoice_date": ["Invoice Date", "Bill Date", "Dated"],
        "due_date": ["Due Date", "Payment Due Date"],
        "po_number": [
            "PO No.", "PO Number", "Purchase Order No.", "Purchase Order Number",
            "Buyer's Order No.", "Buyers Order No.", "Buyer Order No."
        ],
    }

    for field, aliases in text_mappings.items():
        hit = _candidate_value(candidates, aliases)
        if hit:
            out[field] = str(hit[1]).strip()
            sources[field] = "raw_label_resolver"

    amount_mappings = {
        "subtotal": ["Subtotal", "Subtotal in INR", "Subtotal Amount"],
        "taxable_amount": ["Taxable Value", "Taxable Amount", "Assessable Value", "Taxable in INR"],
        "discount_total": ["Discount Total", "Total Discount"],
        "tax_total": ["Total GST", "Total Tax", "GST Total", "Tax Total"],
        "cgst_total": ["Total CGST", "CGST Total"],
        "sgst_total": ["Total SGST", "SGST Total"],
        "igst_total": ["Total IGST", "IGST Total", "Integrated GST", "Integrated GST Amount"],
        "adjustment": ["Adjustment", "Adj.", "Adj"],
        "shipping_charges": ["Shipping Charges", "Freight Charges"],
        "other_charges": ["Other Charges", "Additional Charges"],
        "round_off": ["Round Off", "Round-off", "Rounding"],
        "total_amount": [
            "Grand Total", "Total Amount", "Total Invoice Value", "Total in INR",
            "Net Payable", "Amount Payable", "Total"
        ],
    }

    for field, aliases in amount_mappings.items():
        hit = _candidate_value(candidates, aliases)
        if hit:
            value = _resolve_amount_candidate(hit)
            if value is not None:
                out[field] = value
                sources[field] = "raw_label_resolver"

    # Explicit IGST rate recovery from labels such as "IGST18(18%)" or "IGST @ 18%".
    if out.get("igst_rate") is None or out.get("igst_total") is None:
        for _nk, label, value, prefix in candidates:
            text = str(label)
            if re.search(r"igst", text, flags=re.I):
                rate_m = re.search(r"(?:@\s*)?(\d+(?:\.\d+)?)\s*%", text)
                if rate_m and out.get("igst_rate") is None:
                    try:
                        out["igst_rate"] = float(rate_m.group(1))
                        sources["igst_rate"] = "raw_label_resolver"
                    except Exception:
                        pass
                amount = _resolve_amount_candidate((label, value, prefix))
                if amount is not None and out.get("igst_total") is None:
                    out["igst_total"] = amount
                    sources["igst_total"] = "raw_label_resolver"
                if out.get("igst_rate") is not None and out.get("igst_total") is not None:
                    break

    party_mappings = {
        "vendor_name": ["Vendor Name", "Vendor", "Supplier Name", "Supplier", "Seller", "Seller Name", "Company Name"],
        "customer_name": [
            "Customer Name", "Customer", "Buyer", "Buyer Name", "Bill To",
            "Billed To", "Sold To", "Party", "M/s."
        ],
        "vendor_address": ["Vendor Address", "Supplier Address", "Seller Address"],
        "customer_address": ["Customer Address", "Buyer Address", "Bill To Address", "Billed To Address", "Party Address"],
        "vendor_gstin": ["Vendor GSTIN", "Supplier GSTIN", "Seller GSTIN", "Vendor GST No.", "GSTIN/UIN"],
        "customer_gstin": ["Customer GSTIN", "Buyer GSTIN", "Bill To GSTIN", "Customer GST No."],
        "vendor_pan": ["Vendor PAN", "Supplier PAN", "Seller PAN", "Company's PAN", "Company’s PAN"],
        "customer_pan": ["Customer PAN", "Buyer PAN"],
        "vendor_phone": ["Vendor Phone", "Phone", "Mobile", "Mobile No.", "Office No."],
        "vendor_email": ["Vendor Email", "Email", "E-Mail", "Email ID"],
    }

    for field, aliases in party_mappings.items():
        hit = _candidate_value(candidates, aliases)
        if hit and hit[1] not in (None, ""):
            value = str(hit[1]).strip()
            out[field] = value.upper() if field.endswith(("gstin", "pan")) else value
            sources[field] = "raw_label_resolver"

    # Explicit Party -> customer fallback for cash-bill style documents.
    if out.get("customer_name") in (None, ""):
        hit = _candidate_value(candidates, ["Party"])
        if hit and hit[1] not in (None, ""):
            out["customer_name"] = str(hit[1]).strip()
            sources["customer_name"] = "raw_label_resolver"

    bd = out.get("bank_details") if isinstance(out.get("bank_details"), dict) else {}

    for field, aliases in {
        "ifsc_code": ["IFSC Code", "IFS Code", "IFSC", "IFS"],
        "account_number": ["Account No.", "Account Number", "A/C No.", "A/C Number", "Bank A/c No."],
        "upi_id": ["UPI ID", "UPI", "VPA"],
        "bank_name": ["Bank Name", "Bank"],
        "branch": ["Branch", "Bank Branch"],
        "account_holder_name": ["Account Name", "Account Holder Name", "Account Holder"],
    }.items():
        hit = _candidate_value(candidates, aliases)
        if hit:
            bd[field] = str(hit[1]).strip()
            sources[f"bank_details.{field}"] = "raw_label_resolver"

    combined = _candidate_value(
        candidates,
        ["Branch & IFS Code", "Branch and IFS Code", "Branch & IFSC Code"]
    )
    if combined:
        text = str(combined[1]).strip()
        ifsc = _extract_ifsc_from_text(text)
        if ifsc:
            bd["ifsc_code"] = ifsc
            sources["bank_details.ifsc_code"] = "raw_label_resolver"

        branch_text = re.sub(
            r"\s*&?\s*(?:IFSC|IFS)\s*(?:CODE)?\s*:?\s*[A-Z]{4}0[A-Z0-9]{6}\b",
            "",
            text,
            flags=re.I,
        ).strip(" &:-")
        if branch_text:
            bd["branch"] = branch_text
            sources["bank_details.branch"] = "raw_label_resolver"

    if bd:
        out["bank_details"] = bd

    # Keep explicit component rates separate from generic GST rate.
    for li in out.get("line_items", []) or []:
        if isinstance(li, dict):
            if li.get("igst_rate") is not None or li.get("cgst_rate") is not None or li.get("sgst_rate") is not None:
                li["gst_rate"] = None

    # No GST/PAN/phone/math validation here.
    return loosely_validate(out), sources, reasons

## Tax type safety
Canonical CGST/SGST/IGST fields are constrained by explicit source tax labels/columns. No tax splitting, conversion, or invention is performed.
 Small extraction fixes: preserve document-level taxable-value labels, preserve numeric zeros, and keep customer/vendor GSTIN fallback values distinct.


In [ ]:
STRUCTURED_VALIDATORS = {
    "vendor_gstin": lambda v: is_valid_gstin_format(v) and is_valid_gstin_checksum(v),
    "customer_gstin": lambda v: is_valid_gstin_format(v) and is_valid_gstin_checksum(v),
    "vendor_pan": is_valid_pan_format,
    "customer_pan": is_valid_pan_format,
    "vendor_cin": lambda v: bool(v) and bool(PATTERNS["vendor_cin"].fullmatch(v.strip().upper())),
    "vendor_phone": lambda v: bool(v) and bool(PATTERNS["vendor_phone"].fullmatch(v.strip())),
    "vendor_email": lambda v: bool(v) and bool(PATTERNS["vendor_email"].fullmatch(v.strip())),
}
REQUIRED_IDENTITY_FIELDS = ["vendor_name", "customer_name", "invoice_number", "total_amount"]


In [ ]:
def extract_document(page_paths: list[str]) -> dict:
    """Extraction-focused entry point.

    This stage intentionally does not run GST/PAN/phone/math/business validations.
    It preserves the model output and returns canonical extraction values so the
    backend validation can be compared separately later.
    """
    t0 = time.time()

    try:
        data, raw_texts = generate_invoice_document_v5(page_paths)
    except Exception as e:
        return {
            "data": None,
            "needs_review": True,
            "review_reasons": [f"generation_error:{type(e).__name__}: {e}"],
            "raw_output": None,
            "generation_path": "generation_error",
            "elapsed_seconds": round(time.time() - t0, 2),
        }

    if data is None:
        return {
            "data": None,
            "needs_review": True,
            "review_reasons": ["model_output_unparseable"],
            "raw_output": raw_texts,
            "generation_path": "unparseable",
            "elapsed_seconds": round(time.time() - t0, 2),
        }

    data, resolved_sources, _ = resolve_invoice_fields(data)

    return {
        "data": data,
        "field_sources": {
            **{f: "llm" for f in CORE_FIELDS if data.get(f) is not None},
            **resolved_sources
        },
        "needs_review": False,
        "review_reasons": [],
        "raw_output": raw_texts,
        "generation_path": "qwen3vl_v5_extraction_only",
        "elapsed_seconds": round(time.time() - t0, 2),
    }


## 12. Test one complete invoice

Upload one PDF or image. The pipeline first detects near-identical duplicate
invoice copies (Original/Duplicate/Triplicate) using image similarity.

Genuine continuation pages are preserved and treated as one logical document.

## What changed in v5.3

1. **Line-item table boundary is stricter.** Summary rows such as Subtotal, IGST, Tax and Total are explicitly rejected.
2. **Tax-inside-row handling.** A service row such as `Professional Fees / IGST @18%` remains one line item; tax goes into tax fields, not the description or line amount.
3. **Financial concepts stay separate.** `taxable_amount`, `subtotal`, `tax_total`, `igst_total`, and `total_amount` are not treated as interchangeable.
4. **HSN/SAC is explicit.** `hsn_sac_code` + `code_type` are available while `hsn_code` remains for compatibility.
5. **Multi-page merge is document-level.** Page 1 establishes identity; all pages contribute line items; only the final page contributes invoice-level totals/bank summary.
6. **Raw generated responses are always preserved.** When JSON is malformed, a failure-only repair attempt is made, but the original response remains available.
7. **Canonical recovery from source labels.** When the model correctly includes a value in `raw_fields`, the extraction layer can map it back into its canonical field without doing business/math validation.

## V5 test runner

Run the upload cell, then page preparation, then the final test cell once.

The final result contains:
- `data`: canonical extraction JSON
- `raw_output`: original Qwen outputs + page-selection metadata
- `field_sources`: field provenance

In [ ]:
from google.colab import files

uploaded = files.upload()
UPLOADED_NAMES = list(uploaded.keys())

print("Uploaded:", UPLOADED_NAMES)

INPUT_PATH = UPLOADED_NAMES[0] if UPLOADED_NAMES else None
print("Selected:", INPUT_PATH)

In [ ]:
import fitz

def prepare_input_pages_v4(input_path: str, dpi: int = 250) -> list[str]:
    ext = os.path.splitext(input_path.lower())[1]

    if ext != ".pdf":
        img = Image.open(input_path).convert("RGB")
        page_path = "/content/invoice_page_1.png"
        img.save(page_path, format="PNG")
        return [page_path]

    doc = fitz.open(input_path)
    paths = []

    try:
        for idx in range(len(doc)):
            pix = doc.load_page(idx).get_pixmap(
                dpi=dpi,
                alpha=False
            )
            page_path = f"/content/invoice_page_{idx + 1}.png"
            pix.save(page_path)
            paths.append(page_path)
    finally:
        doc.close()

    return paths


PAGE_PATHS = prepare_input_pages_v4(INPUT_PATH) if INPUT_PATH else []
print("Prepared pages:", PAGE_PATHS)

In [ ]:
# Run the whole uploaded document as ONE invoice/document.
# IMPORTANT: run this cell only after the upload and page-preparation cells above.

if PAGE_PATHS:
    RESULT = extract_document(PAGE_PATHS)

    print("=" * 75)
    print("ELAPSED:", RESULT.get("elapsed_seconds"), "seconds")
    print("NEEDS REVIEW:", RESULT.get("needs_review"))
    print("REVIEW REASONS:", RESULT.get("review_reasons"))
    print("=" * 75)

    print("\nEXTRACTED DATA:")
    print(json.dumps(
        RESULT.get("data"),
        indent=2,
        ensure_ascii=False
    ))

    print("\nFIELD SOURCES:")
    print(json.dumps(
        RESULT.get("field_sources"),
        indent=2,
        ensure_ascii=False
    ))

    print("\nRAW MODEL OUTPUT KEYS:",
          list((RESULT.get("raw_output") or {}).keys()))
else:
    print("No invoice uploaded. Run the upload and page-preparation cells first.")


In [ ]:
# V5.3 schema sanity check
print("Invoice fields:", list(InvoiceSchema.model_fields.keys()))
assert "igst_rate" in InvoiceSchema.model_fields
assert "igst_total" in InvoiceSchema.model_fields
assert "adjustment" in InvoiceSchema.model_fields
assert "line_amount" in LineItem.model_fields
assert "hsn_sac_code" in LineItem.model_fields
print("V5.3 schema fix: OK")
